In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/credit_risk_data_cleaned.csv')
print(df.head())
print(df.shape)
statistics = pd.DataFrame({
    'Num_of_missing_values': df.isnull().sum(),
    'Data_type': df.dtypes
})
print(statistics)

    client_ID  person_age  person_income person_home_ownership  \
0  CUST_00001          22          59000                  RENT   
1  CUST_00002          21           9600                   OWN   
2  CUST_00003          25           9600              MORTGAGE   
3  CUST_00004          23          65500                  RENT   
4  CUST_00005          24          54400                  RENT   

   person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  \
0              123.0    PERSONAL          D      35000          16.02   
1                5.0   EDUCATION          B       1000          11.14   
2                1.0     MEDICAL          C       5500          12.87   
3                4.0     MEDICAL          C      35000          15.23   
4                8.0     MEDICAL          C      35000          14.27   

   loan_status  ...  city_latitude city_longitude  employment_type  \
0            1  ...        43.6532       -79.3832    Self-employed   
1            0  ...     

In [2]:
df.describe(include = 'all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
client_ID,32581,32581,CUST_00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
person_age,32581.0,NaN,NaN,NaN,27.7346,6.348078,20.0,23.0,26.0,30.0,144.0
person_income,32581.0,NaN,NaN,NaN,66074.84847,61983.119168,4000.0,38500.0,55000.0,79200.0,6000000.0
person_home_ownership,32581,4,RENT,16446,NaN,NaN,NaN,NaN,NaN,NaN,NaN
person_emp_length,32581.0,NaN,NaN,NaN,4.767994,4.087372,0.0,2.0,4.0,7.0,123.0
loan_intent,32581,6,EDUCATION,6453,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_grade,32581,7,A,10777,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,32581.0,NaN,NaN,NaN,9589.371106,6322.086646,500.0,5000.0,8000.0,12200.0,35000.0
loan_int_rate,32581.0,NaN,NaN,NaN,11.00962,3.081611,5.42,8.49,10.99,13.11,23.22
loan_status,32581.0,NaN,NaN,NaN,0.218164,0.413006,0.0,0.0,0.0,0.0,1.0


In [3]:
df['age_group_raw']=pd.qcut(df['person_age'], q=4)
print(df['age_group_raw'].value_counts().sort_index())

age_group_raw
(19.999, 23.0]    8766
(23.0, 26.0]      9063
(26.0, 30.0]      6995
(30.0, 144.0]     7757
Name: count, dtype: int64


In [4]:
age_labels = ['20-23', '23-26', '26-30', '30+']
df['age_group'] = pd.qcut(df['person_age'], q=4, labels=age_labels)
display(df[['person_age', 'age_group_raw', 'age_group']].head())

,person_age,age_group_raw,age_group
0,22,"(19.999, 23.0]",20-23
1,21,"(19.999, 23.0]",20-23
2,25,"(23.0, 26.0]",23-26
3,23,"(19.999, 23.0]",20-23
4,24,"(23.0, 26.0]",23-26


In [5]:
df['credit_history_group_raw'] = pd.qcut(df['cb_person_cred_hist_length'], q=4)
df['credit_history_group']=df['credit_history_group_raw'].cat.rename_categories(['Ngắn', 'Trung bình', 'Dài', 'Rất dài'])
display(df[['cb_person_cred_hist_length', 'credit_history_group_raw', 'credit_history_group']].head())
display(df[['credit_history_group_raw', 'credit_history_group']].value_counts().sort_index())

,cb_person_cred_hist_length,credit_history_group_raw,credit_history_group
0,3,"(1.999, 3.0]",Ngắn
1,2,"(1.999, 3.0]",Ngắn
2,3,"(1.999, 3.0]",Ngắn
3,2,"(1.999, 3.0]",Ngắn
4,4,"(3.0, 4.0]",Trung bình


credit_history_group_raw  credit_history_group
(1.999, 3.0]              Ngắn                    11908
                          Trung bình                  0
                          Dài                         0
                          Rất dài                     0
(3.0, 4.0]                Ngắn                        0
                          Trung bình               5925
                          Dài                         0
                          Rất dài                     0
(4.0, 8.0]                Ngắn                        0
                          Trung bình                  0
                          Dài                      7541
                          Rất dài                     0
(8.0, 30.0]               Ngắn                        0
                          Trung bình                  0
                          Dài                         0
                          Rất dài                  7207
Name: count, dtype: int64

In [6]:
high_dti=df['debt_to_income_ratio'] > 0.4
has_past_default = df['past_delinquencies'] > 0
df['financial_stress_flag'] = np.where(high_dti | has_past_default, 1, 0)

In [7]:
display(df[['person_age', 'age_group', 'cb_person_cred_hist_length', 'credit_history_group', 'financial_stress_flag']].head())

,person_age,age_group,cb_person_cred_hist_length,credit_history_group,financial_stress_flag
0,22,20-23,3,Ngắn,1
1,21,20-23,2,Ngắn,1
2,25,23-26,3,Ngắn,1
3,23,20-23,2,Ngắn,1
4,24,23-26,4,Trung bình,1


In [8]:
df.to_csv('data/credit_risk_data_feature_extracted.csv', index=False)